In [3]:
import os
import csv
from typing import Literal
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from PIL import Image
from tqdm import tqdm

import random

seed = 42

random.seed(seed)                  # Python built-in random
np.random.seed(seed)               # NumPy
torch.manual_seed(seed)            # PyTorch (CPU)
torch.cuda.manual_seed(seed)       # PyTorch (single GPU)
torch.cuda.manual_seed_all(seed)   # PyTorch (all GPUs)

# Ensures deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [4]:

#-------------读取训练集,训练集地址已经设定好，下面这段不用修改------------------#
#-----Read the training set, the address of the training set has been set, and the following section does not need to be modified-------#
train_path = 'sum_prod/' if os.name == 'nt' else "/bohr/train-i5ob/v1"


In [10]:

def get_table():
    res = {}
    for a in range(10):
        for b in range(10):
            for c in range(10):
                for d in range(10):
                    state = a + b + c + d, a * b * c * d
                    if state not in res:
                        res[state] = []
                    res[state].append((a, b, c, d))
    result = [[[True] * 10 for _ in range(6562)] for _ in range(37)]
    for k, v in res.items():
        for perm in v:
            for val in perm:
                result[k[0]][k[1]][val] = False
    return result

raw_answers = '''
3495,0908,9450,6278,5727,7113,3724,9234,1693,4056,
8259,0787,9527,4286,8566,6167,4636,9208,6798,8618,
1100,8466,8089,2094,3807,1211,2780,2314,3576,1091,
4326,2295,1378,9466,5031,1200,7109,9925,4455,2917,
7451,1504,5394,4541,7366,5501,3215,2047,5059,9487,
1186,6779,3852,5720,2819,9553,3915,6318,2711,9062,
5814,9992,5310,3339,0800,7744,4742,2147,7931,9118,
0609,2883,7367,3051,0539,5054,3114,7826,0079,6838,
2793,3945,1310,3733,7344,7041,8158,0225,8999,7244,
5459,5329,7635,5055,5328,6254,4864,0741,3837,1791
'''.replace('\n', '').split(',')
raw_target = '000001,21,540;000002,17,0;000003,18,0;000004,23,672;000005,21,490;000006,12,21;000007,16,168;000008,18,216;000009,19,162;000010,15,0;000011,24,720;000012,22,0;000013,23,630;000014,20,384;000015,25,1440;000016,14,0;000017,19,432;000018,19,0;000019,30,3024;000020,23,384;000021,2,0;000022,24,1152;000023,25,0;000024,15,0;000025,18,0;000026,5,2;000027,19,224;000028,10,24;000029,21,630;000030,11,0;000031,15,144;000032,18,180;000033,19,168;000034,25,1296;000035,9,0;000036,3,0;000037,17,0;000038,25,810;000039,18,400;000040,19,126;000041,19,210;000042,10,0;000043,21,540;000044,16,120;000045,22,756;000046,11,0;000047,11,30;000048,13,0;000049,19,0;000050,28,2016;000051,16,48;000052,29,2646;000053,18,240;000054,14,0;000055,20,144;000056,22,675;000057,18,135;000058,19,192;000059,11,14;000060,17,0;000061,18,160;000062,29,1458;000063,9,0;000064,18,243;000065,8,0;000066,22,784;000067,17,224;000068,22,504;000069,20,189;000070,19,72;000071,15,0;000072,21,384;000073,23,882;000074,9,0;000075,17,0;000076,14,0;000077,9,12;000078,23,672;000079,16,0;000080,25,1152;000081,21,378;000082,21,540;000083,5,0;000084,16,189;000085,18,336;000086,12,0;000087,22,320;000088,9,0;000089,35,5832;000090,17,224;000091,21,540;000092,19,270;000093,21,630;000094,15,0;000095,18,240;000096,17,240;000097,22,768;000098,12,0;000099,21,504;000100,18,63'
raw_target = raw_target.split(';')

for i in range(len(raw_answers)):
    ts, tp = map(int, raw_target[i].split(',')[1:])
    ds = tuple(map(int, raw_answers[i]))
    s, p = sum(ds), ds[0] * ds[1] * ds[2] * ds[3]
    if s != ts or p != tp:
        print(i, s, p, ts, tp)
breakpoint()

# 读取数据。
def load_train_data(data_dir='./train/'):
    label_path = os.path.join(data_dir, 'train_labels.csv')
    image_dir = os.path.join(data_dir, 'train_images')

    data = []
    with open(label_path, 'r', newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            sample_id = row['id']
            data.append({
                'id': sample_id,
                'image_path': os.path.join(image_dir, f'{sample_id}.png'),
                'sum': float(row['sum']),
                'product': float(row['product']),
            })

    print(f'Successfully loaded training records: {len(data)}')
    return data


class MNISTBaselineDataset(Dataset):

    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    @staticmethod
    def load_image(image_path):
        image = Image.open(image_path).convert('L')
        image = np.array(image, dtype=np.float32) / 255.0
        # image = (image - 0.1307) / 0.3081
        return image

    def __getitem__(self, idx):
        record = self.records[idx]
        image = self.load_image(record['image_path'])
        image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)
        target = torch.tensor([record['sum'], record['product']], dtype=torch.float32)
        return image, target


def create_train_loader(batch_size=64, num_workers=1, data_dir='./train/'):
    records = load_train_data(data_dir)
    dataset = MNISTBaselineDataset(records)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
    )


class SimpleCNN(nn.Module):

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 2),
        )

    def forward(self, x):
        # torch.Size([64, 1, 28, 112])
        x = self.features(x)
        return self.regressor(x)

class MyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.convs = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.GELU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.GELU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.GELU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.head = nn.Sequential(
            nn.Linear(64, 32),
            nn.GELU(),
            nn.Linear(32, 10)
        )
    
    def forward(self, x: torch.Tensor, mode: Literal['train', 'eval', 'raw']):
        (batch_size, _, _, _) = x.shape
        digits = torch.concat(tuple(x[:, :, :, 28 * i: 28 * i + 28] for i in range(4)))
        conv_output = self.convs(digits)
        preds: torch.Tensor = self.head(conv_output.reshape((batch_size * 4, 64)))
        result = torch.reshape(preds, (4, batch_size, 10)).permute((1, 0, 2))
        if mode == 'raw':
            return result
        if mode == 'train':
            raw_assumption = result.softmax(-1)
            assumption = torch.mul(raw_assumption, torch.linspace(0, 9, 10, device=device).unsqueeze(0).unsqueeze(0)).sum(-1)
            pred_sum = assumption.sum(-1)
            pred_prod = (assumption.prod(-1) + 1).log()
            return torch.stack((pred_sum, pred_prod)).permute((1, 0)), raw_assumption
        assumption = torch.argmax(result, -1)
        pred_sum = assumption.sum(-1)
        pred_prod = assumption.prod(-1)
        return torch.stack((pred_sum, pred_prod)).permute((1, 0))

# 训练模型
def train(model, train_loader, epochs, device='cpu'):
    model.to(device)
    criterion = nn.MSELoss()
    criterion2 = nn.CrossEntropyLoss()
    criterion3 = nn.BCELoss()
    # optimizer = optim.SGD(model.parameters(), lr=0.01)
    optimizer = optim.AdamW(model.parameters(), lr=0.01)

    table = get_table()
    # s = 0
    # for u in table:
    #     for v in u:
    #         s += sum(v)
    # print(s, 'in', len(table),len(table[0]),len(table[0][0]))

    def pad_idx(x: int, d: int):
        x = str(x)
        return '0' * (d - len(x)) + x

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        # save me
        for i in range(len(raw_answers)):
            image = MNISTBaselineDataset.load_image(os.path.join(train_path, f'train_images/{pad_idx(i + 1, 6)}.png'))
            image = torch.tensor(image, dtype=torch.float32, device=device).unsqueeze(0).unsqueeze(0)
            targets = torch.tensor(list(map(int, raw_answers[i])), dtype=torch.int64, device=device)
            optimizer.zero_grad()
            preds = model(image, mode='raw').squeeze(0)
            loss = criterion2(preds, targets)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f'avg fit: {total_loss / len(raw_answers)}')

        total_loss = 0.0
        total_imp = 0
        total_entropy = 0
        for images, targets in tqdm(train_loader, desc=f'Epoch {epoch}/{epochs}', leave=False):
            images = images.to(device)
            
            raw_targets: list[list[float]] = targets.tolist()
            impossible: list[list[bool]] = []
            for pair in raw_targets:
                impossible.append(table[int(pair[0])][int(pair[1])])
            impossible = torch.tensor(impossible, dtype=torch.float32, device=device).unsqueeze(1)

            targets = targets.to(device)
            targets[:, 1] = torch.log(targets[:, 1] + 1)

            optimizer.zero_grad()
            preds, prob = model(images, mode='train')
            
            conflict = torch.mul(prob, impossible)
            
            loss = criterion(preds, targets) + criterion3(conflict, torch.zeros((len(raw_targets), 4, 10), device=device)) * 30
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * images.size(0)
            total_imp += impossible.sum().item()

            nums = prob.shape[0] * 4
            flat_prob = torch.reshape(prob, (nums, 10))
            entropy = torch.mul(flat_prob, torch.log(flat_prob))
            total_entropy += (entropy.sum() / nums).item()

        avg_loss = total_loss / len(train_loader.dataset)
        print(f'Epoch {epoch}/{epochs} - Loss: {avg_loss:.4f} - imp per dig: {total_imp / len(train_loader.dataset):.4f} entropy: {total_entropy / len(train_loader.dataset)}')


def set_random_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


device = 'cuda' if torch.cuda.is_available() else 'cpu'
seed = 42
batch_size = 64
epochs = 32

set_random_seed(seed)
train_loader = create_train_loader(batch_size=batch_size, data_dir=train_path)
# model = SimpleCNN()
model = MyCNN()
train(model, train_loader, epochs=epochs, device=str(device))


In [ ]:

#-------------读取测试集---------------#“DATA_PATH”是测试集加密后的环境变量，按照如下方式可以在提交后，系统评分时访问测试集，但是选手无法直接下载
#----Read the testing set, “DATA_PATH” is an environment variable for the encrypted test set. After submission, you can access the test set for system scoring in the following manner, but the contestant cannot download it directly.-----#
if os.environ.get('DATA_PATH'):
    test_path = os.environ.get("DATA_PATH") + "/"
else:
    test_path = "./test/"
    print("Baseline 运行时，因为无法读取测试集，所以会有此条报错，属于正常现象")
    print("When baseline is running, this error message will appear because the test set cannot be read, which is a normal phenomenon.")
    #Baseline 运行时，因为无法读取测试集，所以会有此条报错，属于正常现象
    #When baseline is running, this error message will appear because the test set cannot be read, which is a normal phenomenon.


In [ ]:

# 读取测试数据
class MNISTTestDataset(Dataset):
    def __init__(self, image_dir):
        self.image_paths = sorted(str(path) for path in Path(image_dir).glob('*.png'))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        sample_id = Path(image_path).stem
        image = MNISTBaselineDataset.load_image(image_path)
        image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)
        return image, sample_id


# 这里用训练好的模型直接回归 sum 和 product。
def predict_and_save(model, image_dir, output_file, device='cpu', batch_size=64, num_workers=1):
    dataset = MNISTTestDataset(image_dir)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
    )

    rows = []
    model.eval()
    with torch.no_grad():
        for images, sample_ids in tqdm(loader, desc=f'Predicting {os.path.basename(image_dir)}', leave=False):
            images = images.to(device)
            preds = model(images, mode='eval').cpu()
            sum_preds = preds[:, 0].round().clamp(0, 36).to(torch.int64).numpy()
            product_preds = preds[:, 1].round().clamp(0, 6561).to(torch.int64).numpy()

            for sample_id, sum_pred, product_pred in zip(sample_ids, sum_preds, product_preds):
                rows.append({
                    'id': sample_id,
                    'sum': int(sum_pred),
                    'product': int(product_pred),
                })

    with open(output_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['id', 'sum', 'product'])
        writer.writeheader()
        writer.writerows(rows)

    print(f'Saved predictions to {output_file}')


val_dir = os.path.join(test_path, 'val')
test_dir = os.path.join(test_path, 'test')

for required_dir in [val_dir, test_dir]:
    if not os.path.isdir(required_dir):
        raise FileNotFoundError(f'Missing test directory: {required_dir}')

predict_and_save(model, val_dir, 'submission_val.csv', device=str(device), batch_size=batch_size)
predict_and_save(model, test_dir, 'submission_test.csv', device=str(device), batch_size=batch_size)



In [ ]:

import zipfile

# 定义要打包的文件和压缩文件名
files_to_zip = ['submission_val.csv', 'submission_test.csv']
zip_filename = 'submission.zip'

# 创建一个 zip 文件
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        # 将文件添加到 zip 文件中
        zipf.write(file, os.path.basename(file))

print(f'{zip_filename} 创建成功!')


